
# Dataset EDA

Exploratory analysis of the NLBSE'26 code comment classification datasets (Java, Python, Pharo).


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path('dataset')
LANG_LABELS = {
    'java': ['summary','Ownership','Expand','usage','Pointer','deprecation','rational'],
    'python': ['Usage','Parameters','DevelopmentNotes','Expand','Summary'],
    'pharo': ['Keyimplementationpoints','Example','Responsibilities','Intent','Keymessages','Collaborators']
}
print(f"Using pandas {pd.__version__}")

Using pandas 2.3.3


In [2]:
java_train_df = pd.read_parquet(DATA_DIR / 'java_train.parquet')
java_test_df = pd.read_parquet(DATA_DIR / 'java_test.parquet')
python_train_df = pd.read_parquet(DATA_DIR / 'python_train.parquet')
python_test_df = pd.read_parquet(DATA_DIR / 'python_test.parquet')
pharo_train_df = pd.read_parquet(DATA_DIR / 'pharo_train.parquet')
pharo_test_df = pd.read_parquet(DATA_DIR / 'pharo_test.parquet')
print('Java train/test:', java_train_df.shape, java_test_df.shape)
print('Python train/test:', python_train_df.shape, python_test_df.shape)
print('Pharo train/test:', pharo_train_df.shape, pharo_test_df.shape)

Java train/test: (5394, 6) (1201, 6)
Python train/test: (1368, 6) (290, 6)
Pharo train/test: (900, 6) (208, 6)


In [3]:
datasets = {}
overview_rows = []
for lang, labels in LANG_LABELS.items():
    datasets[lang] = {}
    for split in ['train','test']:
        path = DATA_DIR / f"{lang}_{split}.parquet"
        df = pd.read_parquet(path)
        datasets[lang][split] = df
        avg_words = df['comment_sentence'].str.split().apply(len).mean()
        overview_rows.append({'language': lang, 'split': split, 'rows': len(df), 'avg_words': round(avg_words, 2)})
        print(f"{lang:<6} {split:<5}: {len(df):4d} rows, columns: {list(df.columns)}")
overview_df = pd.DataFrame(overview_rows)
print('Overview (rows & avg words per sentence):')
print(overview_df)

java   train: 5394 rows, columns: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels']
java   test : 1201 rows, columns: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels']
python train: 1368 rows, columns: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels']
python test :  290 rows, columns: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels']
pharo  train:  900 rows, columns: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels']
pharo  test :  208 rows, columns: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels']
Overview (rows & avg words per sentence):
  language  split  rows  avg_words
0     java  train  5394      10.61
1     java   test  1201      12.66
2   python  train  1368       6.99
3   python   test   290       7.71
4    pharo  train   900       9.15
5    pharo   test   208       8.61


In [4]:
label_tables = {}
for lang, labels in LANG_LABELS.items():
    rows = []
    for split in ['train','test']:
        df = datasets[lang][split]
        arr = np.array(df['labels'].tolist())
        # Create a fresh copy of labels for each split
        current_labels = list(labels)
        if arr.shape[1] != len(current_labels):
            # pad label names if needed
            extra = arr.shape[1] - len(current_labels)
            current_labels = current_labels + [f'label_{i}' for i in range(extra)]
        rows.append(pd.Series(arr.sum(axis=0), index=current_labels, name=split))
    table = pd.DataFrame(rows).astype(int)
    label_tables[lang] = table
    print(f"{lang.upper()} label distribution (counts):")
    print(table)

JAVA label distribution (counts):
       summary  Ownership  Expand  usage  Pointer  deprecation  rational
train     2535        188     377   1515      675           80       221
test       618         28      79    295      125           10        58
PYTHON label distribution (counts):
       Usage  Parameters  DevelopmentNotes  Expand  Summary
train    431         402               167     279      255
test      91          85                32      51       61
PHARO label distribution (counts):
       Keyimplementationpoints  Example  Responsibilities  Intent  \
train                      126      374               180     117   
test                        28       89                42      21   

       Keymessages  Collaborators  
train          163             50  
test            30              7  


In [5]:
length_summary = {}
for lang in LANG_LABELS:
    stats = []
    for split in ['train','test']:
        df = datasets[lang][split]
        lengths = df['comment_sentence'].str.split().apply(len) 
        stats.append({
            'split': split,
            'min': int(lengths.min()),
            'median': float(lengths.median()),
            'mean': round(float(lengths.mean()), 2),
            'p90': float(lengths.quantile(0.9)),
            'max': int(lengths.max())
        })
    length_summary[lang] = pd.DataFrame(stats)
    print(f"{lang.upper()} sentence length stats (word counts):")
    print(length_summary[lang])

JAVA sentence length stats (word counts):
   split  min  median   mean   p90  max
0  train    1     7.0  10.61  20.0  288
1   test    1     8.0  12.66  28.0  154
PYTHON sentence length stats (word counts):
   split  min  median  mean   p90  max
0  train    1     7.0  6.99  12.0   26
1   test    1     8.0  7.71  13.0   21
PHARO sentence length stats (word counts):
   split  min  median  mean   p90  max
0  train    1     8.0  9.15  19.0   47
1   test    1     7.0  8.61  18.0   34


In [6]:
multilabel_summary = {}
for lang in LANG_LABELS:
    stats = []
    for split in ['train','test']:
        df = datasets[lang][split]
        arr = np.array(df['labels'].tolist())
        label_counts = arr.sum(axis=1)
        stats.append({
            'split': split,
            'single_label': int((label_counts == 1).sum()),
            'multi_label': int((label_counts > 1).sum()),
            'max_labels_in_sentence': int(label_counts.max())
        })
    multi_df = pd.DataFrame(stats)
    multi_df['multi_ratio'] = (multi_df['multi_label'] / (multi_df['single_label'] + multi_df['multi_label'])).round(3)
    multilabel_summary[lang] = multi_df
    print(f"{lang.upper()} single vs multi-label counts:")
    print(multi_df)

JAVA single vs multi-label counts:
   split  single_label  multi_label  max_labels_in_sentence  multi_ratio
0  train          5212          182                       3        0.034
1   test          1189           12                       2        0.010
PYTHON single vs multi-label counts:
   split  single_label  multi_label  max_labels_in_sentence  multi_ratio
0  train          1211          157                       3        0.115
1   test           262           28                       3        0.097
PHARO single vs multi-label counts:
   split  single_label  multi_label  max_labels_in_sentence  multi_ratio
0  train           801           99                       3        0.110
1   test           199            9                       2        0.043


In [7]:
TOP_K = 5
top_classes = {}
for lang in LANG_LABELS:
    df = datasets[lang]['train']
    counts = df['class'].value_counts().head(TOP_K)
    top_classes[lang] = counts
    print(f"{lang.upper()} top {TOP_K} classes by labeled sentences:")
    print(counts)

JAVA top 5 classes by labeled sentences:
class
NNThroughputBenchmark.java    163
SmoothRateLimiter.java        161
PaintTarget.java               80
ICElement.java                 75
JobHistoryParser.java          69
Name: count, dtype: int64
PYTHON top 5 classes by labeled sentences:
class
Retry                57
PlotAccessor         53
BCEWithLogitsLoss    48
EmbeddingBag         41
Unfold               40
Name: count, dtype: int64
PHARO top 5 classes by labeled sentences:
class
BlLayout                  46
SpartaCanvas              26
MooseEntity               22
GtGraphTreemapSquarify    20
BrExamplesShowroom        19
Name: count, dtype: int64


In [8]:
# peek at a few representative comments per language
for lang in LANG_LABELS:
    df = datasets[lang]['train']
    sample = df.sample(3, random_state=42)[['class','comment_sentence','labels']]
    print(f"Sample {lang} sentences:")
    print(sample.to_string(index=False))

Sample java sentences:
                       class                                                                                  comment_sentence                labels
CSourcePreviewerUpdater.java \t * Registers a source preview updater for the given viewer, configuration and preference store. [1, 0, 0, 0, 0, 0, 0]
  TextViewerDragAdapter.java                                                                     @param viewer the text viewer [0, 0, 0, 1, 0, 0, 0]
              ICElement.java                                                                         \t * C++ template method. [0, 0, 1, 0, 0, 0, 0]
Sample python sentences:
            class                                                                  comment_sentence          labels
        ExcelFile                                             see read excel for more documentation [0, 0, 1, 0, 0]
BCEWithLogitsLoss                                   if attr reduction is none , then math n, , same [0, 1, 0, 1, 0]
       F

## Quick Takeaways

- **Java**: 7614 training sentences; `summary` dominates (47.4% of samples). Multi-label share ≈ 2.4%. Avg words/sentence ≈ 10.53.
- **Python**: 1884 training sentences; `Usage` dominates (30.7% of samples). Multi-label share ≈ 8.3%. Avg words/sentence ≈ 6.97.
- **Pharo**: 1298 training sentences; `Example` dominates (42.1% of samples). Multi-label share ≈ 10.2%. Avg words/sentence ≈ 9.07.

> Pharo label vectors contain a seventh position not documented in the README; it's labeled `Unknown` above (76 train, 10 test sentences).

In [9]:
java_train_df

,index,class,comment_sentence,partition,combo,labels
0,1,Abfss.java,this impl delegates to the old filesystem,0,this impl delegates to the old filesystem | Ab...,"[0, 0, 1, 0, 0, 0, 0]"
1,2,AbstractContractGetFileStatusTest.java,test getfilestatus and related listing operati...,0,test getfilestatus and related listing operati...,"[1, 0, 0, 0, 0, 0, 0]"
2,15,AbstractContractGetFileStatusTest.java,a filesystem filter which exposes the protecte...,0,a filesystem filter which exposes the protecte...,"[0, 0, 1, 0, 0, 0, 0]"
3,16,AbstractContractGetFileStatusTest.java,"@link #listlocatedstatus path, pathfilter .",0,"@link #listlocatedstatus path, pathfilter . | ...","[0, 0, 1, 0, 0, 0, 0]"
4,18,ApplicationConstants.java,the type of launch for the container.,0,the type of launch for the container. | Applic...,"[0, 0, 1, 0, 0, 0, 0]"
...,...,...,...,...,...,...
5389,10648,TestProcessCorruptBlocks.java,// wait for one minute for deletion to...,0,// wait for one minute for deletion to...,"[1, 0, 0, 0, 0, 0, 0]"
5390,10649,TestProcessCorruptBlocks.java,"// Each datanode has multiple data dirs, c...",0,"// Each datanode has multiple data dirs, c...","[1, 0, 0, 0, 0, 0, 0]"
5391,10651,TestProcessCorruptBlocks.java,// wait for 3 seconds so that all block ...,0,// wait for 3 seconds so that all block ...,"[1, 0, 0, 0, 0, 0, 0]"
5392,10653,TestProcessCorruptBlocks.java,// wait for 3 seconds so that all block ...,0,// wait for 3 seconds so that all block ...,"[1, 0, 0, 0, 0, 0, 0]"


In [10]:
java_test_df

,index,class,comment_sentence,partition,combo,labels
0,5,AbstractContractGetFileStatusTest.java,accept everything.,1,accept everything. | AbstractContractGetFileSt...,"[0, 0, 1, 0, 0, 0, 0]"
1,8,AbstractContractGetFileStatusTest.java,accept nothing.,1,accept nothing. | AbstractContractGetFileStatu...,"[0, 0, 1, 0, 0, 0, 0]"
2,12,AbstractContractGetFileStatusTest.java,equals the @code match field.,1,equals the @code match field. | AbstractContra...,"[0, 0, 1, 0, 0, 0, 0]"
3,19,ApplicationConstants.java,environment for applications.,1,environment for applications. | ApplicationCon...,"[0, 0, 1, 0, 0, 0, 0]"
4,23,BalancingPolicy.java,balancing policy.,1,balancing policy. | BalancingPolicy.java,"[1, 0, 0, 0, 0, 0, 0]"
...,...,...,...,...,...,...
1196,4243,GDBBreakpoints_7_0.java,\t\t\t// Passcount is just for tracepoints,1,\t\t\t// Passcount is just for tracepoints | G...,"[1, 0, 0, 0, 0, 0, 0]"
1197,4255,ProcessClosure.java,Contributors:\n * IBM Corporation - initi...,1,Contributors:\n * IBM Corporation - initi...,"[0, 1, 0, 0, 0, 0, 0]"
1198,4309,IASTSimpleDeclSpecifier.java,Contributors:\n * Doug Schaefer (IBM) - In...,1,Contributors:\n * Doug Schaefer (IBM) - In...,"[0, 1, 0, 0, 0, 0, 0]"
1199,4375,CImperativeSymbolTable.java,@author Mike Kucera,1,@author Mike Kucera | CImperativeSymbolTable....,"[0, 1, 0, 0, 0, 0, 0]"


In [11]:
pharo_test_df

,index,class,comment_sentence,partition,combo,labels
0,5,BlArrowheadSimpleArrow,users can also customise nose angle that tells...,1,users can also customise nose angle that tells...,"[0, 1, 0, 0, 0, 0]"
1,6,BlArrowheadSimpleArrow,the with of the outer arrows can be specified ...,1,the with of the outer arrows can be specified ...,"[0, 0, 1, 0, 0, 0]"
2,7,BlArrowheadSimpleArrow,i support both background and border paint and...,1,i support both background and border paint and...,"[0, 0, 1, 0, 0, 0]"
3,9,BlArrowheadTriangle,my size depends on the width of a curve and he...,1,my size depends on the width of a curve and he...,"[0, 0, 1, 0, 0, 0]"
4,11,BlArrowheadTriangle,i support background and border paints.,1,i support background and border paints. | BlAr...,"[0, 0, 1, 0, 0, 0]"
...,...,...,...,...,...,...
203,1812,WAPlugin,to add a new plugin make sure you choose the r...,1,to add a new plugin make sure you choose the r...,"[0, 0, 1, 0, 0, 0]"
204,1814,WAProtectionFilter,the protection filter ensures that the wrapped...,1,the protection filter ensures that the wrapped...,"[0, 0, 1, 0, 0, 0]"
205,1816,WARedirectingRegistry,i revert to the old 33.0 behavior which is eas...,1,i revert to the old 33.0 behavior which is eas...,"[0, 0, 1, 0, 0, 0]"
206,1818,WARequestContext,it does not matter if this is a request to a s...,1,it does not matter if this is a request to a s...,"[0, 0, 1, 0, 0, 0]"


The list of labels is defined for each of as follows:
- `java`: [`summary`, `Ownership`, `Expand`, `usage`, `Pointer`, `deprecation`, `rational`],
- `python`: [`Usage`, `Parameters`, `DevelopmentNotes`, `Expand`, `Summary`],
- `pharo`: [`Keyimplementationpoints`, `Example`, `Responsibilities`, `Intent`, `Keymessages`, `Collaborators`]